In [1]:
#第8章/加载数据集
from datasets import load_dataset
import torchvision
import torch


def get_dataset():
    #加载数据集
    dataset = load_dataset('lansinuote/gen.5.flower.book', split='train')

    #删除多余的字段
    dataset = dataset.remove_columns(['cls'])

    #图像数据预处理
    compose = torchvision.transforms.Compose([
        torchvision.transforms.Resize(64),
        torchvision.transforms.ToTensor(),
        lambda x: x * 2 - 1,
    ])

    def f(data):
        image = compose(data['image'][0]).unsqueeze(dim=0)
        return {'image': image}

    dataset = dataset.with_transform(f)

    #为了加速数据遍历的效率,把全体数据载入内存以加速IO
    dataset_tensor = torch.empty(len(dataset), 3, 64, 64)

    for i in range(len(dataset)):
        dataset_tensor[i] = dataset[i]['image']

    return dataset_tensor


dataset = get_dataset()

dataset.shape, dataset.dtype

Using custom data configuration lansinuote--gen.5.flower.book-34a531175c385260
Found cached dataset parquet (/root/.cache/huggingface/datasets/lansinuote___parquet/lansinuote--gen.5.flower.book-34a531175c385260/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


(torch.Size([2000, 3, 64, 64]), torch.float32)

In [2]:
#第8章/定义loader
loader = torch.utils.data.DataLoader(dataset=dataset,
                                     batch_size=64,
                                     shuffle=True,
                                     drop_last=True)

len(loader), next(iter(loader)).shape

(31, torch.Size([64, 3, 64, 64]))

In [3]:
#第8章/show函数
def show(images):
    from matplotlib import pyplot as plt

    images = images.to('cpu').detach()[:50]
    images = images.permute(0, 2, 3, 1)
    images = (images + 1) / 2

    plt.figure(figsize=(20, 10))

    for i in range(len(images)):
        plt.subplot(5, 10, i + 1)
        plt.imshow(images[i])
        plt.axis('off')

    plt.show()


show(next(iter(loader)))

<Figure size 2000x1000 with 50 Axes>

In [4]:
#第8章/定义工具子层
class Block(torch.nn.Module):

    def __init__(self, dim_in, dim_out, is_encoder=True):
        super().__init__()

        cnn_type = torch.nn.Conv2d
        if not is_encoder:
            cnn_type = torch.nn.ConvTranspose2d

        def block(dim_in, dim_out, kernel_size=3, stride=1, padding=1):
            return (
                cnn_type(dim_in,
                         dim_out,
                         kernel_size=kernel_size,
                         stride=stride,
                         padding=padding),
                torch.nn.BatchNorm2d(dim_out),
                torch.nn.LeakyReLU(),
            )

        self.s = torch.nn.Sequential(
            *block(dim_in, dim_in),
            *block(dim_in, dim_in),
            *block(dim_in, dim_in),
            *block(dim_in, dim_out, kernel_size=3, stride=2, padding=0),
            *block(dim_out, dim_out),
            *block(dim_out, dim_out),
            *block(dim_out, dim_out),
        )

        self.res = cnn_type(dim_in,
                            dim_out,
                            kernel_size=3,
                            stride=2,
                            padding=0)

    def forward(self, x):
        return self.s(x) + self.res(x)


Block(3, 5)(torch.randn(2, 3, 20, 20)).shape

torch.Size([2, 5, 9, 9])

In [5]:
#第8章/定义Encoder模型
encoder = torch.nn.Sequential(
    Block(3, 32, True),
    Block(32, 64, True),
    Block(64, 128, True),
    Block(128, 256, True),
    torch.nn.Flatten(),
    torch.nn.Linear(2304, 128),
)

encoder(torch.randn(2, 3, 64, 64)).shape

torch.Size([2, 128])

In [6]:
#第8章/定义Decoder模型
decoder = torch.nn.Sequential(
    torch.nn.Linear(128, 256 * 4 * 4),
    torch.nn.InstanceNorm1d(256 * 4 * 4),
    torch.nn.Unflatten(dim=1, unflattened_size=(256, 4, 4)),
    Block(256, 128, False),
    Block(128, 64, False),
    Block(64, 32, False),
    Block(32, 3, False),
    torch.nn.UpsamplingNearest2d(size=64),
    torch.nn.Conv2d(in_channels=3,
                    out_channels=3,
                    kernel_size=1,
                    stride=1,
                    padding=0),
    torch.nn.Tanh(),
)

decoder(torch.randn(2, 128)).shape

torch.Size([2, 3, 64, 64])

In [7]:
#第8章/初始化工具类
optimizer = torch.optim.Adam(decoder.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer,
                                              start_factor=1,
                                              end_factor=0,
                                              total_iters=1000 * len(loader))
criterion = torch.nn.MSELoss()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder.to(device)
decoder.to(device)

encoder.train()
decoder.train()

device

'cuda'

In [8]:
#第8章/训练
def train():
    for epoch in range(1000):
        for _, data in enumerate(loader):
            data = data.to(device)

            pred = decoder(encoder(data))
            loss = criterion(pred, data)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        if epoch % 50 == 0:
            print(epoch, loss.item(), optimizer.param_groups[0]['lr'])

            with torch.no_grad():
                gen = decoder(torch.randn(10, 128, device=device))
            show(gen)

    torch.save(encoder.to('cpu'), 'save/encoder.model')
    torch.save(decoder.to('cpu'), 'save/decoder.model')


train()

0 0.1599009484052658 0.00019979999999999992


<Figure size 2000x1000 with 10 Axes>

50 0.06995858997106552 0.00018979999999999567


<Figure size 2000x1000 with 10 Axes>

100 0.04789628088474274 0.00017979999999999136


...

<Figure size 2000x1000 with 10 Axes>

900 0.026749659329652786 1.9799999999993763e-05


<Figure size 2000x1000 with 10 Axes>

950 0.02764085680246353 9.799999999996925e-06


<Figure size 2000x1000 with 10 Axes>

In [9]:
#第8章/测试
decoder = torch.load('save/decoder.model')

with torch.no_grad():
    gen = decoder(torch.randn(50, 128))

show(gen)

<Figure size 2000x1000 with 50 Axes>

In [10]:
#第8章/在线加载笔者训练好的模型并测试
from transformers import PreTrainedModel, PretrainedConfig


#包装类
class Model(PreTrainedModel):
    config_class = PretrainedConfig

    def __init__(self, config):
        super().__init__(config)
        self.encoder = encoder.to('cpu')
        self.decoder = decoder.to('cpu')
        
#加载训练好的模型
decoder = Model.from_pretrained('lansinuote/gen.1.ae.book').decoder

with torch.no_grad():
    gen = decoder(torch.randn(50, 128))

show(gen)

<Figure size 2000x1000 with 50 Axes>